# Week 4, day 2 (afternoon) — dbt on Snowflake: from raw to data marts

The WeCloudData **Create a dbt Project** lab and the **dbt Fundamentals**
lecture, run entirely inside Snowflake.

You create the warehouse, database, schemas, raw tables, file format and stages;
load the data; then deploy a complete dbt project — sources, staging, snapshots,
a **Type 6** slowly-changing dimension, a **star schema** on surrogate keys,
seeds that answer real business requirements, custom generic tests, unit tests
and a **MetricFlow semantic layer** — and run it as a native `DBT PROJECT`
object.

## What is where

| | |
|---|---|
| **This notebook** | every Snowflake statement: create objects, load, deploy, run dbt, inspect, schedule, tear down. All SQL. |
| **`dbt-project/demo/`** | the dbt project as real files, in the repo. You upload this folder to a stage. Read its `README.md` first. |

**There are no credentials here.** A Snowflake notebook is already
authenticated, and dbt runs as the role that executes the project.

## Before you start

1. A role that can create a database, schemas, stages, tasks and a `DBT PROJECT`,
   with a warehouse attached to this notebook.
2. The two CSVs to hand — `products.csv` (1,214 rows) and `sales.csv`
   (100,000 rows). Question 7 uploads them.
3. The `dbt-project/demo/` folder to hand. Question 9 uploads it.

## The layers

```
RAW  ──▶  STG  ──▶  EDW  ──▶  MARTS
 │         │         │          │
 │         │         │          ├─ rpt_*   hand-written SQL models
 │         │         │          └─ mart_*  exported from MetricFlow saved queries
 │         │         └─ dim_product_t6 (Type 6) / dim_store / dim_date / fct_sales
 │         └─ stg_product_incr / stg_sales (views) + the snapshots
 └─ product / sales  (the two CSVs)      + seeds: store_master, category_targets
```

New to dbt? The day's `README.md` opens with a concepts primer — model, source,
ref, seed, snapshot, materialization, macro, test, and the three ways to
configure a model. Read that first.

Work the questions in order: each depends on what the earlier ones made.


---

## Instructor notes

**Cells are in dependency order** — run top to bottom.

### Before the session

1. **Upload both artefacts once and keep them.** The two CSVs, and the
   `dbt-project/demo/` folder. Question 9 expects ~36 files under
   `@DEMO_DB.RAW.DBT_PROJECT_STAGE/demo/`, with `dbt_project.yml` at the root of
   that prefix.
2. **Dry-run the whole thing.** The first `build` takes a few minutes on 100k
   rows.
3. **Do not add packages.** `packages.yml` is empty on purpose — Snowflake has no
   outbound internet by default, and a listed-but-uninstalled package makes every
   dbt command fail before it runs anything. The project's README explains what
   replaced them.

### The manual steps

Everything else is a cell you run. These are not:

| Question | You do it by hand |
|---|---|
| 7 | upload the two CSVs to `RAW.LOAD_STAGE` |
| 9 | upload `dbt-project/demo/` to `RAW.DBT_PROJECT_STAGE` |
| 10 | re-upload + re-run `CREATE OR REPLACE DBT PROJECT` after any file change |

### Beats worth slowing down on

- **Question 15** — the Type 6 dimension, three views on one row.
- **Question 16** — why `prod_key` cannot key an SCD.
- **Question 19** — change a product, re-snapshot, rebuild, watch the columns
  diverge. The payoff; do not skip it for time.
- **Questions 20–21** — lineage. Ask the room "what breaks if I change
  `stg_sales`?" before running it.
- **Question 23** — `GET_DDL`. Nobody wrote `CREATE TABLE`; here it is anyway.

### Timing

Part A (~20 min) is object creation — brisk. Part B is the deploy. Part C is the
substance; budget half the session. Parts D–E are ~10 minutes each. Leave five
minutes for the teardown.

### About the output

This notebook ships **without stored output** — it was written against no
Snowflake account, so nothing here is a recorded run. Each answer states what it
*should* return. Execute it once yourself before class so the outputs are yours.


Run this first, every session.

In [ ]:
-- Run this first. It sets the session context every later cell assumes.
-- There are NO credentials in this notebook: a Snowflake notebook is already
-- authenticated, and dbt runs as the role executing the project.
USE ROLE SYSADMIN;              -- or whichever role owns the lab objects
USE WAREHOUSE COMPUTE_WH;       -- change if your warehouse is named differently
USE DATABASE DEMO_DB;

SELECT CURRENT_ROLE()      AS my_role,
       CURRENT_WAREHOUSE() AS my_warehouse,
       CURRENT_DATABASE()  AS my_database,
       CURRENT_VERSION()   AS snowflake_version;

## PART A — create the objects

Everything the lab needs, made explicitly. dbt creates objects *inside* these; it
never creates the database, the schemas or the raw tables.

### Question 1

Create the **warehouse**, and check your role can do what follows.
Skip the `CREATE` if you already have one.

In [ ]:
-- XSMALL is plenty for 100k rows. AUTO_SUSPEND keeps the credit burn near zero
-- between cells, and INITIALLY_SUSPENDED means creating it costs nothing.
CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH
    WAREHOUSE_SIZE      = 'XSMALL'
    AUTO_SUSPEND        = 60
    AUTO_RESUME         = TRUE
    INITIALLY_SUSPENDED = TRUE;

USE WAREHOUSE COMPUTE_WH;

SHOW GRANTS TO ROLE IDENTIFIER(CURRENT_ROLE());

A warehouse left running bills by the second, so `AUTO_SUSPEND` matters in a classroom. If `SHOW GRANTS` does not include `CREATE DATABASE`, switch to a role that has it before going further.

### Question 2

**Optional, ACCOUNTADMIN only.** A dedicated least-privilege role for the
lab. Skip it if you are running as `SYSADMIN`.

In [ ]:
USE ROLE ACCOUNTADMIN;

CREATE ROLE IF NOT EXISTS DBT_LAB_ROLE;

GRANT USAGE, OPERATE ON WAREHOUSE COMPUTE_WH TO ROLE DBT_LAB_ROLE;
GRANT CREATE DATABASE ON ACCOUNT             TO ROLE DBT_LAB_ROLE;
GRANT ROLE DBT_LAB_ROLE TO USER IDENTIFIER(CURRENT_USER());
GRANT ROLE DBT_LAB_ROLE TO ROLE SYSADMIN;

SHOW GRANTS TO ROLE DBT_LAB_ROLE;

Doing coursework as ACCOUNTADMIN is a habit worth breaking. Whichever role you settle on is the one dbt runs as, so it must be able to create objects in every schema below.

### Question 3

Create the database and the five schemas — one per layer.

In [ ]:
-- RAW    the landing zone: the two CSVs, untouched
-- STG    staging (1:1 with raw) and the snapshots
-- EDW    the star schema: conformed dimensions and facts
-- MARTS  business-facing reports
-- SEED   dbt seeds (version-controlled reference data)
CREATE DATABASE IF NOT EXISTS DEMO_DB;
USE DATABASE DEMO_DB;

CREATE SCHEMA IF NOT EXISTS RAW;
CREATE SCHEMA IF NOT EXISTS STG;
CREATE SCHEMA IF NOT EXISTS EDW;
CREATE SCHEMA IF NOT EXISTS MARTS;
CREATE SCHEMA IF NOT EXISTS SEED;

SHOW SCHEMAS IN DATABASE DEMO_DB;

One schema per layer, matching the folder-to-schema routing in the project's `dbt_project.yml`. That mapping is what makes the lineage readable and lets you build a layer at a time.

### Question 4

Create the two raw tables, matching the CSV headers exactly.

In [ ]:
USE SCHEMA RAW;

CREATE OR REPLACE TABLE RAW.PRODUCT (
    PROD_KEY          NUMBER,
    PROD_NAME         VARCHAR,
    VOL               FLOAT,
    WGT               FLOAT,
    BRAND_NAME        VARCHAR,
    STATUS_CODE       NUMBER,
    STATUS_CODE_NAME  VARCHAR,
    CATEGORY_KEY      NUMBER,
    CATEGORY_NAME     VARCHAR,
    SUBCATEGORY_KEY   NUMBER,
    SUBCATEGORY_NAME  VARCHAR
);

CREATE OR REPLACE TABLE RAW.SALES (
    TRANS_ID     NUMBER,
    PROD_KEY     NUMBER,
    STORE_KEY    NUMBER,
    TRANS_DT     DATE,
    TRANS_TIME   NUMBER,
    PRIORITY     VARCHAR,
    SALES_QTY    FLOAT,
    SALES_PRICE  FLOAT,
    SALES_AMT    FLOAT,
    DISCOUNT     FLOAT,
    SALES_COST   FLOAT,
    SALES_MGRN   FLOAT,
    SHIPMODE     VARCHAR,
    SHIP_COST    FLOAT
);

SHOW TABLES IN SCHEMA RAW;

Declare the types deliberately rather than letting a load wizard infer them. `TRANS_DT` must be a real `DATE` — the date dimension and every time-based metric depend on it.

### Question 5

Create a named **file format**, so the parsing rules are one object every
`COPY` reuses.

In [ ]:
CREATE OR REPLACE FILE FORMAT RAW.CSV_FF
    TYPE = CSV
    SKIP_HEADER = 1                        -- both files have a header row
    FIELD_DELIMITER = ','
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'     -- text values are quoted
    DATE_FORMAT = 'MM/DD/YYYY'             -- TRANS_DT is written M/D/YYYY
    NULL_IF = ('', 'NULL')
    EMPTY_FIELD_AS_NULL = TRUE;

DESC FILE FORMAT RAW.CSV_FF;

`DATE_FORMAT` is the one that bites. `TRANS_DT` is `M/D/YYYY` (e.g. `3/7/2010`); without it every date whose day and month are both ≤ 12 is silently misread rather than rejected — a wrong answer instead of an error.

### Question 6

Create the two **stages**: one for the CSVs, one for the dbt project.

In [ ]:
-- The CSVs you upload.
CREATE STAGE IF NOT EXISTS RAW.LOAD_STAGE
    DIRECTORY = (ENABLE = TRUE)
    FILE_FORMAT = RAW.CSV_FF;

-- The dbt project folder you upload.
CREATE STAGE IF NOT EXISTS RAW.DBT_PROJECT_STAGE
    DIRECTORY = (ENABLE = TRUE)
    ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE');

SHOW STAGES IN DATABASE DEMO_DB;

Both stages live in `RAW`. `SNOWFLAKE_SSE` encryption on the project stage matters: a `DBT PROJECT` object can only be created from a server-side-encrypted stage.

### Question 7

**Upload the two CSVs** to `RAW.LOAD_STAGE`, then load them.

In Snowsight: **Data → Databases → DEMO_DB → RAW → Stages → LOAD_STAGE →
+ Files**. Then run this.

In [ ]:
-- Confirm what actually landed before copying.
LIST @RAW.LOAD_STAGE;

COPY INTO RAW.PRODUCT FROM @RAW.LOAD_STAGE/products.csv
    FILE_FORMAT = RAW.CSV_FF
    ON_ERROR = 'ABORT_STATEMENT';

COPY INTO RAW.SALES FROM @RAW.LOAD_STAGE/sales.csv
    FILE_FORMAT = RAW.CSV_FF
    ON_ERROR = 'ABORT_STATEMENT';

`LIST` first: if a client compressed on upload the names gain `.gz`, and the `COPY` path must match. `ON_ERROR = 'ABORT_STATEMENT'` makes a bad row fail loudly instead of being skipped silently.

### Question 8

Verify the load — row counts, and the date range that proves the format
was applied.

In [ ]:
SELECT 'product' AS table_name, COUNT(*) AS row_count FROM RAW.PRODUCT
UNION ALL
SELECT 'sales', COUNT(*) FROM RAW.SALES;

-- Must span 2009-01-01 to 2012-12-30. NULLs or a wrong range mean the date
-- format did not apply -- TRUNCATE and load again.
SELECT MIN(TRANS_DT) AS first_day,
       MAX(TRANS_DT) AS last_day,
       COUNT(DISTINCT TRANS_DT) AS distinct_days
FROM RAW.SALES;

Expect **1,214** products and **100,000** sales. A row count only proves arrival; the date range proves the parse. Checking both is the habit worth forming.

## PART B — deploy and run the dbt project

The project is a folder of files in this repo at
`week4_day2_afternoon/dbt-project/demo/`. You upload it to the stage, turn it
into a `DBT PROJECT` object, and run dbt as SQL.

Read that folder's `README.md` before uploading — it lists exactly what to
include, and explains why `packages.yml` is empty.

### Question 9

**Upload the project folder** to `@DEMO_DB.RAW.DBT_PROJECT_STAGE/demo/`,
preserving its directory structure, then confirm what landed.

`dbt_project.yml` must sit at the root of that prefix. Exclude `target/`,
`logs/` and `profiles.example.yml`.

In [ ]:
LIST @DEMO_DB.RAW.DBT_PROJECT_STAGE;

-- You are looking for demo/dbt_project.yml at the top, then the folders:
--   demo/macros/  demo/models/  demo/seeds/  demo/snapshots/
--   demo/tests/   demo/analyses/  demo/profiles.yml  demo/packages.yml
SELECT COUNT(*) AS files_staged
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

Snowsight's stage browser can upload a folder tree; SnowSQL can `PUT` with `AUTO_COMPRESS = FALSE` (dbt must read plain `.sql` and `.yml`, not `.gz`). Expect around 36 files. If `dbt_project.yml` is nested one level too deep, the next statement fails to find a project.

### Question 10

Create the **dbt project object** from the stage.

In [ ]:
CREATE OR REPLACE DBT PROJECT DEMO_DB.PUBLIC.SALES_DBT
    FROM @DEMO_DB.RAW.DBT_PROJECT_STAGE/demo/;

SHOW DBT PROJECTS IN DATABASE DEMO_DB;

The project is now a database object you can grant on and schedule. It is a **snapshot of the stage, not a live link** — re-upload and re-run this `CREATE OR REPLACE` after every change to a model.

### Question 11

Build everything: seeds, snapshots, models and tests, in dependency order.

In [ ]:
-- No `deps` step: this project depends on NO packages, on purpose. Snowflake
-- has no outbound internet by default, so a packages.yml listing an uninstalled
-- package fails every command with
--   "found N package(s) specified in packages.yml, but only 0 installed".
-- The three things a package would have given us are written by hand in the
-- project -- see its README.

EXECUTE DBT PROJECT DEMO_DB.PUBLIC.SALES_DBT ARGS = 'build';

`build` runs seeds → snapshots → models → tests in DAG order and **stops a downstream model when an upstream test fails**, so bad data does not propagate. That is why it is the command you schedule. (`run` = models only, `snapshot` = snapshots only, `test` = tests only.)

### Question 12

Run one layer at a time, one model, and a full refresh — the same
selectors as the CLI.

In [ ]:
EXECUTE DBT PROJECT DEMO_DB.PUBLIC.SALES_DBT ARGS = 'run --select staging';
EXECUTE DBT PROJECT DEMO_DB.PUBLIC.SALES_DBT ARGS = 'run --select edw';
EXECUTE DBT PROJECT DEMO_DB.PUBLIC.SALES_DBT ARGS = 'run --select marts';

EXECUTE DBT PROJECT DEMO_DB.PUBLIC.SALES_DBT ARGS = 'test --select dim_product_t6';

-- Rebuild incrementals from scratch, ignoring the is_incremental() cutoff.
-- Needed whenever a model's SQL or schema changes.
EXECUTE DBT PROJECT DEMO_DB.PUBLIC.SALES_DBT ARGS = 'build --full-refresh';

Because each layer is a folder, the folder name doubles as a layer selector. Building layer by layer is how you debug a broken DAG without waiting for the whole thing.

### Question 13

Materialize the semantic layer's saved queries — the extra marts.

In [ ]:
EXECUTE DBT PROJECT DEMO_DB.PUBLIC.SALES_DBT ARGS = 'build --select saved_query:*';

SHOW TABLES IN SCHEMA DEMO_DB.MARTS;

Three more marts, declared from metric definitions rather than written as SQL. Note the MetricFlow `mf` CLI (`mf query`, `mf list metrics`) is a *local* tool and is not available inside Snowflake — the semantic models still parse and the exports still build.

## PART C — inspect what dbt built

The interesting part. Everything below is a plain query against objects dbt
created.

### Question 14

Look at staging and the two snapshots.

In [ ]:
SELECT * FROM DEMO_DB.STG.STG_SALES LIMIT 10;

-- The snapshot's own history columns. PRODUCT_SNAPSHOT uses the check strategy,
-- SALES_SNAPSHOT the timestamp strategy -- same columns, different trigger.
SELECT prod_key, dbt_valid_from, dbt_valid_to
FROM DEMO_DB.STG.PRODUCT_SNAPSHOT
ORDER BY prod_key LIMIT 10;

`dbt_valid_to` is null on the current version of each row. That single column is what the Type 6 dimension turns into three different answers.

### Question 15

Look at the **Type 6** dimension — all three views on one row.

In [ ]:
SELECT product_sk,
       prod_key,
       category_name           AS type2_as_of_then,
       current_category_name   AS type1_today,
       previous_category_name  AS type3_changed_from,
       valid_from, valid_to, is_current
FROM DEMO_DB.EDW.DIM_PRODUCT_T6
ORDER BY prod_key, valid_from
LIMIT 20;

Type 6 = 1 + 2 + 3. On a first build every product has one version, so all three columns agree — the last question in this part makes them diverge.

### Question 16

Prove the **surrogate key** is what makes the dimension addressable.

In [ ]:
SELECT COUNT(*)                    AS rows,
       COUNT(DISTINCT product_sk)  AS distinct_surrogate_keys,
       COUNT(DISTINCT prod_key)    AS distinct_natural_keys
FROM DEMO_DB.EDW.DIM_PRODUCT_T6;

`rows` equals `distinct_surrogate_keys` always. It equals `distinct_natural_keys` only until a product gains a second version — which is exactly why `prod_key` cannot key an SCD, and why the fact joins on `product_sk`.

### Question 17

Query the **star**: the fact joined to all three dimensions.

In [ ]:
SELECT d.year_num,
       d.month_name,
       s.region,
       p.current_category_name AS category,
       SUM(f.sales_amt) AS sales_amt,
       SUM(f.sales_qty) AS sales_qty
FROM DEMO_DB.EDW.FCT_SALES f
JOIN DEMO_DB.EDW.DIM_DATE       d ON f.date_key   = d.date_key
JOIN DEMO_DB.EDW.DIM_PRODUCT_T6 p ON f.product_sk = p.product_sk
JOIN DEMO_DB.EDW.DIM_STORE      s ON f.store_sk   = s.store_sk
GROUP BY 1, 2, 3, 4
ORDER BY 1, 2, 5 DESC
LIMIT 20;

-- Orphan check: every fact row must find its dimensions.
SELECT COUNT(*) AS orphan_rows
FROM DEMO_DB.EDW.FCT_SALES f
LEFT JOIN DEMO_DB.EDW.DIM_PRODUCT_T6 p ON f.product_sk = p.product_sk
WHERE p.product_sk IS NULL;

Four tables, three joins, all on surrogate keys — a star schema doing its job. `orphan_rows` must be 0; the `relationships` tests in the project assert the same thing automatically on every build.

### Question 18

Look at the marts, including the one the **seed** exists for.

In [ ]:
SELECT * FROM DEMO_DB.MARTS.RPT_CATEGORY_VS_TARGET ORDER BY pct_of_target DESC;

SELECT * FROM DEMO_DB.MARTS.RPT_SALES_BY_REGION LIMIT 10;

-- Same question, two routes: SQL somebody wrote vs metrics somebody declared.
-- If these disagree, the hand-written model has drifted from the agreed
-- definition -- which is the argument for the semantic layer.
SELECT 'hand-written'      AS source, ROUND(SUM(sales_amt)) AS sales
FROM DEMO_DB.MARTS.RPT_SALES_BY_REGION
UNION ALL
SELECT 'metricflow export', ROUND(SUM(total_sales))
FROM DEMO_DB.MARTS.MART_SALES_BY_REGION_MONTHLY;

`RPT_CATEGORY_VS_TARGET` compares actuals to a planning number that exists in no source system — it came from a seed in the repo. That is the whole point of seeds.

### Question 19

**Make history happen.** Change a product, re-snapshot, rebuild, and watch
the Type 6 columns diverge.

In [ ]:
-- 1. Change a product in the source.
UPDATE DEMO_DB.RAW.PRODUCT
SET CATEGORY_NAME = 'category-5'
WHERE PROD_KEY = (SELECT MIN(PROD_KEY) FROM DEMO_DB.RAW.PRODUCT);

-- 2. Capture it and rebuild.
EXECUTE DBT PROJECT DEMO_DB.PUBLIC.SALES_DBT ARGS = 'snapshot';
EXECUTE DBT PROJECT DEMO_DB.PUBLIC.SALES_DBT ARGS = 'build';

-- 3. Two rows now, for the same product.
SELECT product_sk, prod_key,
       category_name          AS type2_as_of_then,
       current_category_name  AS type1_today,
       previous_category_name AS type3_changed_from,
       valid_from, valid_to, is_current
FROM DEMO_DB.EDW.DIM_PRODUCT_T6
WHERE prod_key = (SELECT MIN(PROD_KEY) FROM DEMO_DB.RAW.PRODUCT)
ORDER BY valid_from;

This is the payoff. The old row keeps its original `category_name` (Type 2) but gains a `valid_to` and `is_current = false`; **both** rows now show `category-5` in `current_category_name` (Type 1); and the new row's `previous_category_name` names what it changed from (Type 3). One dimension, three questions answered.

## PART D — lineage across the layers

dbt knows the whole graph because every model declares its inputs with `ref()`
and `source()`. Both directions are answerable — and the second is the question
to ask *before* editing anything.

### Question 20

Trace **upstream**: everything that feeds the regional sales report.

In [ ]:
EXECUTE DBT PROJECT DEMO_DB.PUBLIC.SALES_DBT
    ARGS = 'ls --select +rpt_sales_by_region';

The `+` prefix means "and everything upstream". You should see the whole chain — sources, the snapshot, staging, seeds, and the edw dimensions and fact: raw → stg → edw → marts in one list.

### Question 21

Trace **downstream**: the blast radius if `stg_sales` changed. Then what a
dashboard depends on.

In [ ]:
EXECUTE DBT PROJECT DEMO_DB.PUBLIC.SALES_DBT
    ARGS = 'ls --select stg_sales+';

EXECUTE DBT PROJECT DEMO_DB.PUBLIC.SALES_DBT
    ARGS = 'ls --select +exposure:sales_dashboard';

A trailing `+` walks the other way. This is the payoff of never hardcoding a table name: the graph answers impact questions a folder of numbered SQL scripts cannot. The exposure puts a dashboard on that graph even though dbt does not build it.

### Question 22

Confirm the layers landed where the project's routing said they would.

In [ ]:
SELECT table_schema, table_name, table_type, row_count
FROM DEMO_DB.INFORMATION_SCHEMA.TABLES
WHERE table_schema IN ('RAW','SEED','STG','EDW','MARTS')
ORDER BY CASE table_schema
             WHEN 'RAW' THEN 1 WHEN 'SEED' THEN 2 WHEN 'STG' THEN 3
             WHEN 'EDW' THEN 4 ELSE 5 END,
         table_name;

The physical proof of the lineage: staging as **views**, edw and marts as **tables**, each in its own schema — exactly the folder-to-schema routing in `dbt_project.yml`. `row_count` is null for views, because they store nothing.

## PART E — what dbt actually wrote, scheduling, and cleanup

### Question 23

Ask Snowflake for the **DDL** of what dbt built.

In [ ]:
-- A staging model: dbt wrapped a SELECT in CREATE VIEW.
SELECT GET_DDL('VIEW', 'DEMO_DB.STG.STG_SALES') AS staging_view_ddl;

-- A dimension: the same idea, materialized as a table.
SELECT GET_DDL('TABLE', 'DEMO_DB.EDW.DIM_PRODUCT_T6') AS type6_dimension_ddl;

-- Or a whole layer at once.
SELECT GET_DDL('SCHEMA', 'DEMO_DB.EDW') AS whole_edw_layer_ddl;

The punchline of the day: nobody wrote `CREATE TABLE`, yet here is the exact DDL. The materialization decided the shape — `STG_SALES` came out a **view**, `DIM_PRODUCT_T6` a **table** — from one config line, not from writing two different statements.

### Question 24

Schedule it. Inside Snowflake the scheduler is already there — a `TASK`,
no cron and no external orchestrator.

In [ ]:
CREATE OR REPLACE TASK DEMO_DB.PUBLIC.SALES_DBT_DAILY
    WAREHOUSE = COMPUTE_WH
    SCHEDULE = 'USING CRON 0 6 * * * UTC'
AS
    EXECUTE DBT PROJECT DEMO_DB.PUBLIC.SALES_DBT ARGS = 'build';

-- Tasks are created SUSPENDED.
ALTER TASK DEMO_DB.PUBLIC.SALES_DBT_DAILY RESUME;

SHOW TASKS IN SCHEMA DEMO_DB.PUBLIC;

Because the task runs `build`, a failing test stops the run and surfaces in task history rather than silently publishing bad data. Change `COMPUTE_WH` if your warehouse differs.

### Question 25

Check the task history — how you find out a scheduled run failed.

In [ ]:
SELECT name, state, scheduled_time, completed_time, error_message
FROM TABLE(DEMO_DB.INFORMATION_SCHEMA.TASK_HISTORY(
        TASK_NAME => 'SALES_DBT_DAILY'))
ORDER BY scheduled_time DESC
LIMIT 20;

`state` and `error_message` are what monitoring watches. A resumed task keeps running on schedule — suspend it when the class ends, which the next cell does.

### Question 26

**Tear it down** — so you can re-run the class from scratch, and so nothing
keeps billing.

In [ ]:
-- 1. STOP THE CLOCK FIRST. A resumed task can fire against half-dropped objects.
ALTER TASK IF EXISTS DEMO_DB.PUBLIC.SALES_DBT_DAILY SUSPEND;

-- 2. Drop the lab objects.
DROP TASK        IF EXISTS DEMO_DB.PUBLIC.SALES_DBT_DAILY;
DROP DBT PROJECT IF EXISTS DEMO_DB.PUBLIC.SALES_DBT;

-- 3. The database takes the schemas, tables, views, stages and file format with
--    it. Comment this out if you want to keep the results.
DROP DATABASE IF EXISTS DEMO_DB;

-- 4. Suspending the warehouse is usually enough; drop only if you made it here.
ALTER WAREHOUSE IF EXISTS COMPUTE_WH SUSPEND;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;

-- 5. And the role, if you created one.
-- USE ROLE ACCOUNTADMIN;
-- DROP ROLE IF EXISTS DBT_LAB_ROLE;

SHOW DATABASES LIKE 'DEMO_DB';

Order matters: suspend the task **before** dropping what it reads. Dropping the database removes the most in one statement. Snowflake keeps it in Time Travel for the retention period, so `UNDROP DATABASE DEMO_DB` recovers an accident.